# Frozen GloVe ablation

We check how much fine-tuning the GloVe word embeddings helps the Stage 1 model. We use the model without lemma fallback, so fine-tuning the embeddings is the only difference between the two models.

## Setup

These cells are copied from `main_revised.ipynb` (every code cell before its first training cell), so both models use the same data, seed and scoring.

In [1]:
import os

if os.path.basename(os.getcwd()) == "ablations":
    os.chdir("..")      # run from the repo root, where the data and models are

In [2]:
import os, re, math, copy, time, zipfile, io, collections, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torchcrf import CRF

# Loads python packages for natural language processing
import nltk
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [3]:
DATASET_FILE = "en-US.jsonl"

# Load the dataset file
DATA_FOLDER = "data"
MODEL_FOLDER = "model"              # all checkpoints are saved here
LOG_FOLDER = "logs"                # training logs (best epochs, per-epoch history) are saved here
os.makedirs(MODEL_FOLDER, exist_ok=True)
os.makedirs(LOG_FOLDER, exist_ok=True)
if not os.path.exists(os.path.join(DATA_FOLDER, DATASET_FILE)):
    raise FileNotFoundError(f"Could not find {os.path.join(DATA_FOLDER, DATASET_FILE)}")
print("Using dataset at:", os.path.join(DATA_FOLDER, DATASET_FILE))

# RETRAIN = True trains every model from scratch. With False, each training cell loads its
# saved checkpoint (MODEL_FOLDER) and training log (LOG_FOLDER), and only trains if one of them is missing.
RETRAIN = False

SEED = 4002
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Configuration parameters for the model
EMB_DIM_WANTED = 300   # GloVe 6B dimension
HIDDEN         = 256
NUM_LAYERS     = 2
DROPOUT        = 0.3
LR             = 1e-3
WEIGHT_DECAY   = 0.01
LABEL_SMOOTH   = 0.1
WORD_DROPOUT   = 0.25
BATCH_SIZE     = 32
NUM_EPOCHS     = 40
PATIENCE       = 8
GRAD_CLIP      = 5.0


def log_path(checkpoint):
    '''The training log (best epochs, per-epoch history) is saved in LOG_FOLDER, named after its checkpoint.'''
    name = os.path.splitext(os.path.basename(checkpoint))[0]
    return os.path.join(LOG_FOLDER, name + '_log.json')


def save_log(checkpoint, log):
    with open(log_path(checkpoint), 'w', encoding='utf-8') as f:
        json.dump(log, f, indent=1, default=lambda x: x.item())   # .item() for numpy scalars


def load_saved(checkpoint):
    '''Returns the saved training log, or None when the model should be (re)trained.'''
    if RETRAIN or not (os.path.exists(checkpoint) and os.path.exists(log_path(checkpoint))):
        return None
    with open(log_path(checkpoint), encoding='utf-8') as f:
        return json.load(f)


print("Device:", device, "| RETRAIN:", RETRAIN)

Using dataset at: data\en-US.jsonl
Device: cuda | RETRAIN: False


In [4]:
df = pd.read_json(os.path.join(DATA_FOLDER, DATASET_FILE), lines=True)

assert (df["partition"] == "train").sum() == 11514
assert (df["partition"] == "dev").sum()   == 2033
assert (df["partition"] == "test").sum()  == 2974
print("Dataset shape:", df.shape, "| split sizes asserted")

train_df = df[df["partition"] == "train"].copy()
val_df   = df[df["partition"] == "dev"].copy()
test_df  = df[df["partition"] == "test"].copy()

x_train, y_train = train_df["utt"].tolist(), train_df["intent"].tolist()
x_val,   y_val   = val_df["utt"].tolist(),   val_df["intent"].tolist()
print("Training examples:  ", len(train_df))
print("Validation examples:", len(val_df))
print("Test examples:      ", len(test_df))

Dataset shape: (16521, 8) | split sizes asserted
Training examples:   11514
Validation examples: 2033
Test examples:       2974


In [5]:
display(df.head(1))

,id,locale,partition,scenario,intent,utt,annot_utt,worker_id
0,0,en-US,test,alarm,alarm_set,wake me up at five am this week,wake me up at [time : five am] [date : this week],1


In [6]:
zip_path = "glove.6B.zip"

if os.path.exists(zip_path):
    glove = {}
    with zipfile.ZipFile(zip_path) as z, z.open(f"glove.6B.{EMB_DIM_WANTED}d.txt") as f:
        for line in io.TextIOWrapper(f, encoding="utf-8"):
            word, *values = line.rstrip().split(" ")
            glove[word] = np.asarray(values, dtype=np.float32)
    print(f"Loaded GloVe {EMB_DIM_WANTED}d from {zip_path} | vocabulary size: {len(glove)}")
else:
    print(f"{zip_path} not found in the project folder")

Loaded GloVe 300d from glove.6B.zip | vocabulary size: 400001


In [7]:
token_pattern = re.compile(r'\w+|[^\w\s]')


def tokenize_with_offsets(text):
    return [(m.group().lower(), m.start(), m.end()) for m in token_pattern.finditer(text)]


def tokenize(text):
    return [t for t, _, _ in tokenize_with_offsets(text)]

In [8]:
PAD_TOKEN, UNK_TOKEN = "<PAD>", "<UNK>"
PAD_ID, UNK_ID = 0, 1
SLOT_PAD_ID = -100


def token_to_id(token, word_to_id, use_lemma):
    # Known words keep their own ID; only unseen words fall back to their lemma
    if token in word_to_id:
        return word_to_id[token]
    if use_lemma:
        lemma = lemmatize_token(token)
        if lemma in word_to_id:
            return word_to_id[lemma]
    return UNK_ID

In [9]:
lemmatizer = WordNetLemmatizer()


def lemmatize_token(token):
    # Try the noun form first (plurals), then the verb form (tenses)
    lemma = lemmatizer.lemmatize(token, pos='n')
    if lemma == token:
        lemma = lemmatizer.lemmatize(token, pos='v')
    return lemma

In [10]:
intents = sorted(set(y_train))
intent_to_id = {v: i for i, v in enumerate(intents)}
num_classes = len(intent_to_id)

In [11]:
print("Unique intents:", num_classes)
for intent, intent_id in intent_to_id.items():
    print(f"{intent_id:>2}  {intent}")

Unique intents: 60
 0  alarm_query
 1  alarm_remove
 2  alarm_set
 3  audio_volume_down
 4  audio_volume_mute
 5  audio_volume_other
 6  audio_volume_up
 7  calendar_query
 8  calendar_remove
 9  calendar_set
10  cooking_query
11  cooking_recipe
12  datetime_convert
13  datetime_query
14  email_addcontact
15  email_query
16  email_querycontact
17  email_sendemail
18  general_greet
19  general_joke
20  general_quirky
21  iot_cleaning
22  iot_coffee
23  iot_hue_lightchange
24  iot_hue_lightdim
25  iot_hue_lightoff
26  iot_hue_lighton
27  iot_hue_lightup
28  iot_wemo_off
29  iot_wemo_on
30  lists_createoradd
31  lists_query
32  lists_remove
33  music_dislikeness
34  music_likeness
35  music_query
36  music_settings
37  news_query
38  play_audiobook
39  play_game
40  play_music
41  play_podcasts
42  play_radio
43  qa_currency
44  qa_definition
45  qa_factoid
46  qa_maths
47  qa_stock
48  recommendation_events
49  recommendation_locations
50  recommendation_movies
51  social_post
52  social

In [12]:
slot_pattern = re.compile(r'\[\s*([^:\[\]]+?)\s*:\s*([^\[\]]*?)\]')


def parse_annotated_utterance(utt, annot_utt):
    spans, plain_text, last_end = [], '', 0
    for match in slot_pattern.finditer(annot_utt):
        plain_text += annot_utt[last_end:match.start()]
        slot_type, slot_value = match.group(1).strip(), match.group(2)
        start = len(plain_text)
        plain_text += slot_value
        spans.append({'type': slot_type, 'start': start,
                      'end': len(plain_text), 'text': slot_value})
        last_end = match.end()
    plain_text += annot_utt[last_end:]
    if plain_text != utt:
        raise ValueError(f"Slot annotation does not match utterance: {utt!r}")
    return spans

In [13]:
def make_aligned_tokens(utt, spans):
    tags, offsets, prev_slot = [], [], None
    for word, start, end in tokenize_with_offsets(utt):
        current_slot = None
        for i, span in enumerate(spans):
            if start < span['end'] and end > span['start']:
                if start < span['start'] or end > span['end']:
                    raise ValueError(f"A token cuts across a slot boundary: {utt!r}")
                current_slot = i
        if current_slot is None:
            tags.append('O')
        else:
            prefix = 'I' if current_slot == prev_slot else 'B'
            tags.append(f"{prefix}-{spans[current_slot]['type']}")
        offsets.append((start, end))
        prev_slot = current_slot
    return tags, offsets

In [14]:
def prepare_joint_records(split_df):
    records = []
    for row_index, row in split_df.iterrows():
        text = row['utt']
        spans = parse_annotated_utterance(text, row['annot_utt'])
        tags, offsets = make_aligned_tokens(text, spans)
        tokens = tokenize(text)
        if len(tokens) != len(tags):
            raise ValueError(f"Token and slot lengths differ at row {row_index}")
        records.append({'source_row': int(row_index), 'utt': text, 'tokens': tokens,
                        'offsets': offsets, 'gold_spans': spans,
                        'slot_tags': tags, 'intent': row['intent']})
    return records


train_records = prepare_joint_records(train_df)
val_records   = prepare_joint_records(val_df)
test_records  = prepare_joint_records(test_df)

In [15]:
training_slot_types = sorted({s['type'] for r in train_records for s in r['gold_spans']})
slot_tags = ['O']
for t in training_slot_types:
    slot_tags.extend([f'B-{t}', f'I-{t}'])
slot_to_id = {tag: i for i, tag in enumerate(slot_tags)}
id_to_slot = {i: tag for tag, i in slot_to_id.items()}
num_slot_labels = len(slot_to_id)

In [16]:
max_len = max(64, max(len(r['tokens']) for r in train_records))
assert num_classes == 60 and num_slot_labels == 111
print('Intent classes:', num_classes, '| BIO labels:', num_slot_labels, '| max_len:', max_len)

Intent classes: 60 | BIO labels: 111 | max_len: 64


In [17]:
def build_vocab(records):
    # Vocabulary from the given training records only
    vocab = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
    for word in sorted({t for row in records for t in row['tokens']}):
        vocab[word] = len(vocab)
    return vocab


word_to_id = build_vocab(train_records)
print("Vocabulary size:", len(word_to_id))

Vocabulary size: 5146


In [18]:
def build_embedding_matrix(vocab):
    matrix_rng = np.random.default_rng(SEED)
    matrix = matrix_rng.normal(0, 0.1, (len(vocab), EMB_DIM_WANTED)).astype(np.float32)
    matrix[PAD_ID] = 0
    for word, idx in vocab.items():
        if word in glove:
            matrix[idx] = glove[word]
    return matrix


embedding_matrix = build_embedding_matrix(word_to_id)

In [19]:
class JointNLUDataset(Dataset):
    def __init__(self, records, word_to_id, intent_to_id, slot_to_id, max_len, use_lemma=False):
        self.records = records
        self.input_ids, self.attention_masks = [], []
        self.intent_ids, self.slot_ids = [], []
        for row in records:
            words = [token_to_id(w, word_to_id, use_lemma) for w in row['tokens']]
            slots = [slot_to_id[tag] for tag in row['slot_tags']]
            if len(words) > max_len:
                raise ValueError(f"Sentence has {len(words)} tokens, limit is {max_len}")
            padding = max_len - len(words)
            self.input_ids.append(torch.tensor(words + [PAD_ID] * padding))
            self.attention_masks.append(
                torch.tensor([1] * len(words) + [0] * padding, dtype=torch.bool))
            self.intent_ids.append(torch.tensor(intent_to_id[row['intent']]))
            self.slot_ids.append(torch.tensor(slots + [SLOT_PAD_ID] * padding))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        row = self.records[i]
        return {'input_ids': self.input_ids[i], 'attention_mask': self.attention_masks[i],
                'intent_labels': self.intent_ids[i], 'slot_labels': self.slot_ids[i],
                'utt': row['utt'], 'offsets': row['offsets'],
                'gold_spans': row['gold_spans'], 'source_row': row['source_row']}

In [20]:
def joint_collate_fn(samples):
    batch = {}
    for key in ('input_ids', 'attention_mask', 'intent_labels', 'slot_labels'):
        batch[key] = torch.stack([item[key] for item in samples])
    for key in ('utt', 'offsets', 'gold_spans', 'source_row'):
        batch[key] = [item[key] for item in samples]
    return batch

In [21]:
def make_loader(records, vocab, use_lemma=False, shuffle=False, batch_size=None):
    dataset = JointNLUDataset(records, vocab, intent_to_id, slot_to_id, max_len,
                              use_lemma=use_lemma)
    generator = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(dataset, batch_size=batch_size or BATCH_SIZE, shuffle=shuffle,
                      collate_fn=joint_collate_fn, generator=generator)

In [22]:
def lemmatize_records(records):
    # Lemmatize model input tokens only; utt, offsets and gold spans stay original,
    # so slot scoring on character spans is unaffected.
    return [{**row, 'tokens': [lemmatize_token(t) for t in row['tokens']]} for row in records]

In [23]:
def word_frequencies(records, vocab):
    counts = collections.Counter(t for row in records for t in row['tokens'])
    freq = torch.zeros(len(vocab))
    for w, i in vocab.items():
        freq[i] = counts.get(w, 0)
    return freq.to(device)

In [24]:
missing_tokens = [w for w in word_to_id if w not in (PAD_TOKEN, UNK_TOKEN) and w not in glove]

print("Training tokens missing from GloVe:", len(missing_tokens))
print(missing_tokens)

Training tokens missing from GloVe: 181
['altucher', 'apahelp', 'apig', 'applebees', 'badpho', 'bardonia', 'bbcnews', 'beanos', 'beiber', 'beibers', 'bestbuy', 'blahblah', 'bobburgers', 'brexit', 'brightener', 'btaxes', 'burmuda', 'calone', 'capabara', 'capuccino', 'channin', 'chhana', 'cnncom', 'comcastcom', 'comcasts', 'companyname', 'dalls', 'dancom', 'dbell', 'ddlj', 'dejvicka', 'deliveroo', 'dolphines', 'entrapreneur', 'faripir', 'favorited', 'flappy', 'flipkart', 'flooringco', 'fmod', 'foke', 'fongs', 'foodnetworkcom', 'foodpoisoning', 'frubs', 'fulls', 'gooseto', 'grubhub', 'gurpur', 'hackernews', 'happypeoplecom', 'helmsworth', 'hihop', 'hillstead', 'hoizer', 'horkersd', 'hwood', 'icebo', 'impanema', 'indranagar', 'infy', 'ingerman', 'insightbb', 'iroomba', 'jaiho', 'jancy', 'joejoe', 'johncena', 'johndoe', 'jpearsonjessica', 'judawa', 'kamycka', 'kashmirstart', 'keidis', 'krishnarajpuram', 'larrylarry', 'lastfm', 'likexchange', 'lincon', 'lomaine', 'mackensie', 'manchow', 'mar

In [25]:
class JointBiLSTM(nn.Module):
    def __init__(self, embedding_matrix, num_intents, num_slots):
        super().__init__()
        vocab_size, d_model = embedding_matrix.shape
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.embedding.weight.data.copy_(torch.as_tensor(embedding_matrix))
        self.lstm = nn.LSTM(d_model, HIDDEN, NUM_LAYERS, batch_first=True,
                            bidirectional=True, dropout=DROPOUT)
        self.dropout = nn.Dropout(DROPOUT)
        self.slot_classifier = nn.Linear(2 * HIDDEN, num_slots)
        self.intent_classifier = nn.Linear(4 * HIDDEN, num_intents)
        self.crf = CRF(num_slots, batch_first=True)

    def forward(self, input_ids, attention_mask):
        lengths = attention_mask.sum(dim=1)
        x = self.dropout(self.embedding(input_ids))
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True,
                                      enforce_sorted=False)
        h, _ = self.lstm(packed)
        h, _ = pad_packed_sequence(h, batch_first=True, total_length=input_ids.size(1))
        h = self.dropout(h)
        m = attention_mask.unsqueeze(-1)
        max_pool = h.masked_fill(~m, float('-inf')).max(dim=1).values
        mean_pool = (h * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)
        return self.intent_classifier(torch.cat([max_pool, mean_pool], dim=-1)), \
               self.slot_classifier(h)

In [26]:
def tags_to_spans(tags):
    spans, cur, start = set(), None, 0
    for i, t in enumerate(tags):
        if t == 'O':
            if cur:
                spans.add((cur, start, i))
                cur = None
        else:
            prefix, slot_type = t[:1], t[2:]
            if prefix == 'B' or cur != slot_type:
                if cur:
                    spans.add((cur, start, i))
                cur, start = slot_type, i
    if cur:
        spans.add((cur, start, len(tags)))
    return spans


def spans_to_char(spans, offsets):
    # Scoring on character spans keeps the metric independent of tokenization
    return {(t, offsets[s][0], offsets[e - 1][1]) for t, s, e in spans}


def gold_char_spans(gold_spans):
    return {(s['type'], s['start'], s['end']) for s in gold_spans}

In [27]:
def intent_accuracy(predicted, gold):
    """100 x correct intent predictions / number of utterances."""
    return 100 * sum(p == g for p, g in zip(predicted, gold)) / len(gold)


def slot_micro_f1(predicted_spans, gold_spans):
    """F1 on exact (slot type, span) matches pooled over utterances: 100 x 2TP / (2TP + FP + FN)."""
    tp = fp = fn = 0
    for pred, gold in zip(predicted_spans, gold_spans):
        tp += len(gold & pred)
        fp += len(pred - gold)
        fn += len(gold - pred)
    return 0.0 if tp == 0 else 100 * 2 * tp / (2 * tp + fp + fn)

In [28]:
def mask_tokens(ids, mask, rate):
    # Hide random tokens; their labels stay as training targets
    hit = (torch.rand(ids.shape, device=ids.device) < rate) & mask & (ids > UNK_ID)
    return ids.masked_fill(hit, UNK_ID)

In [29]:
def unpack(batch, freq=None, training=False, mask_rate=0.0):
    ids  = batch['input_ids'].to(device)
    mask = batch['attention_mask'].to(device)
    slots = batch['slot_labels'].to(device)
    T = int(mask.sum(dim=1).max())              # trim padding-only columns
    ids, mask, slots = ids[:, :T], mask[:, :T], slots[:, :T]
    if training and WORD_DROPOUT > 0 and freq is not None:
        p = WORD_DROPOUT / (WORD_DROPOUT + freq[ids])
        hit = (torch.rand_like(p) < p) & mask & (ids > UNK_ID)
        ids = torch.where(hit, torch.full_like(ids, UNK_ID), ids)
    if training and mask_rate > 0:
        ids = mask_tokens(ids, mask, mask_rate)
    return ids, mask, slots, batch['intent_labels'].to(device)

In [30]:
def compute_loss(model, intent_logits, emissions, mask, slots, intent):
    return F.cross_entropy(intent_logits, intent, label_smoothing=LABEL_SMOOTH) \
           - model.crf(emissions, slots.clamp(min=0), mask=mask, reduction='mean')

In [31]:
def train_one_epoch(model, loader, optimizer, scheduler, freq, mask_rate=0.0):
    model.train()
    total_loss, total = 0.0, 0
    for batch in loader:
        ids, mask, slots, intent = unpack(batch, freq, training=True, mask_rate=mask_rate)
        optimizer.zero_grad()
        intent_logits, emissions = model(ids, mask)
        loss = compute_loss(model, intent_logits, emissions, mask, slots, intent)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item() * ids.size(0)
        total += ids.size(0)
    return total_loss / total

In [32]:
def evaluate(model, loader):
    '''Returns (loss, intent accuracy, slot micro-F1) using the shared scoring functions.'''
    model.eval()
    total_loss, total = 0.0, 0
    pred_intents, gold_intents, pred_spans, gold_spans = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            ids, mask, slots, intent = unpack(batch)
            intent_logits, emissions = model(ids, mask)
            total_loss += compute_loss(model, intent_logits, emissions, mask,
                                       slots, intent).item() * ids.size(0)
            total += ids.size(0)
            pred_intents += intent_logits.argmax(dim=1).tolist()
            gold_intents += intent.tolist()
            for path, offsets, gold in zip(model.crf.decode(emissions, mask=mask),
                                           batch['offsets'], batch['gold_spans']):
                pred_spans.append(spans_to_char(tags_to_spans([id_to_slot[i] for i in path]), offsets))
                gold_spans.append(gold_char_spans(gold))
    return (total_loss / total, intent_accuracy(pred_intents, gold_intents),
            slot_micro_f1(pred_spans, gold_spans))

In [33]:
def run_experiment(embedding_matrix, train_loader, val_loaders, train_records_for_freq,
                   vocab, prefix='bilstm', mask_rate=0.0):
    checkpoints = {label: os.path.join(MODEL_FOLDER, f"{prefix}_{label.lower().replace(' ', '_')}_best.pt")
                   for label in val_loaders}
    logs = {label: load_saved(ckpt) for label, ckpt in checkpoints.items()}
    if all(log is not None for log in logs.values()):      # RETRAIN is off and everything is saved
        return load_experiment(embedding_matrix, val_loaders, checkpoints, logs)

    torch.manual_seed(SEED)            # same starting weights for every variant
    model = JointBiLSTM(embedding_matrix, num_classes, num_slot_labels).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps = math.ceil(len(train_loader.dataset) / BATCH_SIZE) * NUM_EPOCHS
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps)
    freq = word_frequencies(train_records_for_freq, vocab)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert total_params <= 20_000_000, f"parameter budget exceeded: {total_params:,}"
    print(f"Total parameters: {total_params:,} "
          f"({100*total_params/20_000_000:.1f}% of the 20M budget)")
    print(f"Trainable parameters: {trainable_params:,}")

    best = {label: {'score': -1.0, 'epoch': None, 'acc': None, 'f1': None, 'state': None}
            for label in val_loaders}
    history = []
    run_start = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        start = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, freq, mask_rate)
        row = {'epoch': epoch, 'train_loss': train_loss}
        for label, loader in val_loaders.items():
            val_loss, acc, f1 = evaluate(model, loader)
            score = (acc + f1) / 2
            row[f'{label} val loss'] = val_loss
            row[f'{label} intent'] = acc
            row[f'{label} slot F1'] = f1
            if score > best[label]['score']:
                best[label] = {'score': score, 'epoch': epoch, 'acc': acc, 'f1': f1,
                               'state': copy.deepcopy(model.state_dict())}
        row['seconds'] = time.time() - start
        history.append(row)
        text = " | ".join(f"{l} val loss {row[f'{l} val loss']:.4f} "
                          f"{row[f'{l} intent']:.2f}%/{row[f'{l} slot F1']:.2f}%"
                          for l in val_loaders)
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train Loss {train_loss:.4f} | "
              f"{text} | {row['seconds']:.1f}s", flush=True)
        if all(epoch - b['epoch'] >= PATIENCE for b in best.values()):
            print(f"Early stopping: no improvement for {PATIENCE} epochs")
            break

    print(f"Training time: {time.time() - run_start:.0f}s over {len(history)} epochs")
    results = {}
    for label, info in best.items():
        filename = checkpoints[label]
        torch.save(info['state'], filename)
        save_log(filename, {'best_epoch': info['epoch'], 'history': history})
        results[label] = {'Best epoch': info['epoch'],
                          'Intent accuracy (%)': round(info['acc'], 2),
                          'Slot micro-F1 (%)': round(info['f1'], 2),
                          'Combined (%)': round(info['score'], 2),
                          'Epochs trained': len(history),
                          'Checkpoint': filename}
        print(f"{label}: best val intent {info['acc']:.2f}% slot F1 {info['f1']:.2f}% "
              f"at epoch {info['epoch']} -> {filename}")
    return model, history, results


def load_experiment(embedding_matrix, val_loaders, checkpoints, logs):
    '''Rebuilds the outputs of run_experiment from saved checkpoints instead of training.'''
    model = JointBiLSTM(embedding_matrix, num_classes, num_slot_labels).to(device)
    results = {}
    for label, loader in val_loaders.items():
        model.load_state_dict(torch.load(checkpoints[label], map_location=device))
        _, acc, f1 = evaluate(model, loader)          # scores are recomputed from the checkpoint
        log = logs[label]
        results[label] = {'Best epoch': log['best_epoch'],
                          'Intent accuracy (%)': round(acc, 2),
                          'Slot micro-F1 (%)': round(f1, 2),
                          'Combined (%)': round((acc + f1) / 2, 2),
                          'Epochs trained': len(log['history']),
                          'Checkpoint': checkpoints[label]}
        print(f"{label}: loaded {checkpoints[label]} (best epoch {log['best_epoch']} of "
              f"{len(log['history'])}) | val intent {acc:.2f}% slot F1 {f1:.2f}%")
    return model, logs[next(iter(val_loaders))]['history'], results

In [34]:
all_results = {}

In [35]:
RETRAIN = False      # set to True to train the frozen model again (the fine-tuned model is always loaded)

train_loader = make_loader(train_records, word_to_id, shuffle=True)
val_loader   = make_loader(val_records, word_to_id, use_lemma=False, batch_size=64)
test_loader  = make_loader(test_records, word_to_id, use_lemma=False, batch_size=64)
print(f"device: {device} | seed: {SEED} | RETRAIN: {RETRAIN}")

device: cuda | seed: 4002 | RETRAIN: False


## Model with fine-tuned GloVe

This model was already trained in `main_revised.ipynb`, so we load it instead of training it again.

In [36]:
TUNED_CHECKPOINT = os.path.join(MODEL_FOLDER, "bilstm_without_fallback_best.pt")

tuned_model = JointBiLSTM(embedding_matrix, num_classes, num_slot_labels).to(device)
tuned_model.load_state_dict(torch.load(TUNED_CHECKPOINT, map_location=device))
with open(log_path(TUNED_CHECKPOINT), encoding="utf-8") as f:
    tuned_log = json.load(f)

_, acc, f1 = evaluate(tuned_model, val_loader)
assert (round(acc, 2), round(f1, 2)) == (86.97, 81.58), "scores differ from main_revised.ipynb"
print(f"Loaded {TUNED_CHECKPOINT} | val intent {acc:.2f}% | val slot F1 {f1:.2f}%")

Loaded model\bilstm_without_fallback_best.pt | val intent 86.97% | val slot F1 81.58%


## Model with frozen GloVe

We keep the GloVe vectors fixed during training, and train the rest of the model with the same settings.

In [37]:
class JointBiLSTMFrozen(JointBiLSTM):
    def __init__(self, *args):
        super().__init__(*args)
        self.embedding.weight.requires_grad_(False)     # the word vectors are not updated

FROZEN_CHECKPOINT = os.path.join(MODEL_FOLDER, "bilstm_without_fallback_frozen_glove_best.pt")


def train_frozen():
    # Follows run_experiment in main_revised.ipynb
    torch.manual_seed(SEED)
    model = JointBiLSTMFrozen(embedding_matrix, num_classes, num_slot_labels).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps = math.ceil(len(train_loader.dataset) / BATCH_SIZE) * NUM_EPOCHS
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps)
    freq = word_frequencies(train_records, word_to_id)

    best, history = {'score': -1.0, 'epoch': None, 'state': None}, []
    for epoch in range(1, NUM_EPOCHS + 1):
        start = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, freq)
        val_loss, acc, f1 = evaluate(model, val_loader)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val loss': val_loss,
                        'intent': acc, 'slot F1': f1, 'seconds': time.time() - start})
        if (acc + f1) / 2 > best['score']:
            best = {'score': (acc + f1) / 2, 'epoch': epoch, 'state': copy.deepcopy(model.state_dict())}
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train Loss {train_loss:.4f} | val {acc:.2f}%/{f1:.2f}%", flush=True)
        if epoch - best['epoch'] >= PATIENCE:
            print(f"Early stopping: no improvement for {PATIENCE} epochs")
            break

    model.load_state_dict(best['state'])
    log = {'best_epoch': best['epoch'], 'history': history}
    torch.save(best['state'], FROZEN_CHECKPOINT)
    save_log(FROZEN_CHECKPOINT, log)
    return model, log


frozen_log = load_saved(FROZEN_CHECKPOINT)
if frozen_log is None:
    frozen_model, frozen_log = train_frozen()
else:
    frozen_model = JointBiLSTMFrozen(embedding_matrix, num_classes, num_slot_labels).to(device)
    frozen_model.load_state_dict(torch.load(FROZEN_CHECKPOINT, map_location=device))
print(f"{FROZEN_CHECKPOINT} | best epoch {frozen_log['best_epoch']} of {len(frozen_log['history'])}")

model\bilstm_without_fallback_frozen_glove_best.pt | best epoch 26 of 34


## Results

In [38]:
results = {}
for name, model, log, frozen in (('Fine-tuned GloVe', tuned_model, tuned_log, False),
                                 ('Frozen GloVe', frozen_model, frozen_log, True)):
    _, val_acc, val_f1 = evaluate(model, val_loader)
    _, test_acc, test_f1 = evaluate(model, test_loader)
    results[name] = {'Val intent (%)': val_acc, 'Val slot F1 (%)': val_f1,
                     'Test intent (%)': test_acc, 'Test slot F1 (%)': test_f1,
                     'Trainable parameters': sum(p.numel() for n, p in model.named_parameters()
                                                 if not (frozen and n == 'embedding.weight')),
                     'Best epoch': log['best_epoch']}

table = pd.DataFrame.from_dict(results, orient='index')
table.loc['Difference'] = table.loc['Fine-tuned GloVe'] - table.loc['Frozen GloVe']
print(table.to_string(float_format=lambda x: f"{x:.2f}"))

                  Val intent (%)  Val slot F1 (%)  Test intent (%)  Test slot F1 (%)  Trainable parameters  Best epoch
Fine-tuned GloVe           86.97            81.58            85.47             78.91               4394530          23
Frozen GloVe               86.67            81.23            85.04             78.66               2850730          26
Difference                  0.30             0.35             0.44              0.25               1543800          -3


Fine-tuning the GloVe vectors improves intent accuracy by 0.30 points and slot F1 by 0.35 points on validation (0.44 and 0.25 points on test), but adds 1,543,800 trainable parameters. The gain is small, so freezing the vectors is a reasonable way to save parameters.